In [2]:
%load_ext autoreload
%autoreload 2

In [157]:
import pandas as pd
from preprocess import format_shop_category_name, format_vendor_name
from sklearn.model_selection import train_test_split
from sklearn.feature_extraction.text import TfidfVectorizer, CountVectorizer
from sklearn.ensemble import RandomForestClassifier
import numpy as np
from sklearn.metrics import f1_score
from sklearn.decomposition import TruncatedSVD
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from pipeline import preprocess_frame, extract_frame

import optuna

In [4]:
RANDOM_STATE = 42

In [33]:
df = pd.read_csv('train.tsv', sep='\t')

In [ ]:
X_train_val, X_test, y_train_val, y_test = train_test_split(df.drop(['category_id', 'department_id'], axis=1),
                                                    df[['category_id', 'department_id']],
                                                    test_size=1/10, shuffle=False)
X_train, X_valid, y_train, y_valid = train_test_split(X_train_val, y_train_val, test_size=1/9, shuffle=False)

In [ ]:
top_vendor_names = format_vendor_name(X_train)["vendor_name"].value_counts().head(11).index.to_list()
top_category_names = format_shop_category_name(X_train)["shop_category_name"].value_counts().head(21).index.to_list()

In [73]:
X_train_pr = preprocess_frame(X_train, top_vendor_names, top_category_names)
X_valid_pr = preprocess_frame(X_valid, top_vendor_names, top_category_names)

In [39]:
char_tfidf = TfidfVectorizer(analyzer="char").fit(X_train_pr["description"])
# char_tfidf_svd = TruncatedSVD(83).fit(char_tfidf.transform(X_train_pr["description"]))
word_tfidf = TfidfVectorizer(analyzer="word").fit(X_train_pr["description"]) # MAX_FEATURES ARGUMENT
word_tfidf_svd = TruncatedSVD(256, random_state=RANDOM_STATE).fit(word_tfidf.transform(X_train_pr["description"]))
char_count = CountVectorizer(analyzer="char").fit(X_train_pr["description"])
# char_count_svd = TruncatedSVD(83).fit(char_count.transform(X_train_pr["description"]))
word_count = CountVectorizer(analyzer="word").fit(X_train_pr["description"])
word_count_svd = TruncatedSVD(256, random_state=RANDOM_STATE).fit(word_count.transform(X_train_pr["description"]))

In [40]:
description_extractors = {
    "char_tfidf": (char_tfidf,),
    "word_tfidf": (word_tfidf, word_tfidf_svd),
    "char_count": (char_count,),
    "word_count": (word_count, word_count_svd),
}

word_tfidf_svd.explained_variance_ratio_.sum(), word_count_svd.explained_variance_ratio_.sum()

(np.float64(0.19568788744777257), np.float64(0.5156678721332025))

In [41]:
vendor_oh = OneHotEncoder().fit(X_train_pr[["vendor_name"]])
cats_oh = OneHotEncoder().fit(X_train_pr[["shop_category_name"]])

In [76]:
X_train_pr = extract_frame(X_train_pr, description_extractors, vendor_oh, cats_oh)
X_valid_pr = extract_frame(X_valid_pr, description_extractors, vendor_oh, cats_oh)

In [77]:
X_train_pr_n = X_train_pr.select_dtypes(include=np.number)
X_valid_pr_n = X_valid_pr.select_dtypes(include=np.number)

In [158]:
to_scale = [col for col in X_train_pr_n.columns if not np.array_equal(np.sort(X_train_pr[col].astype(int).unique()), np.array([0., 1.]))]
scaler = StandardScaler()
X_train_pr_n[to_scale] = scaler.fit_transform(X_train_pr[to_scale])
X_valid_pr_n[to_scale] = scaler.transform(X_valid_pr_n[to_scale])

In [161]:
# weighted_f1_scorer = make_scorer(
#     f1_score, 
#     average="weighted", 
#     greater_is_better=True
# )
    # scores = cross_validate(
    #     model, X_train_pr_n, y_train,
    #     cv=3, 
    #     scoring=weighted_f1_scorer, 
    #     n_jobs=-1,
    #     return_train_score=True
    # )

    # return scores['test_score'].mean()


def objective(trial: optuna.Trial) -> float:
    # params = {
    #     'n_estimators': trial.suggest_int('n_estimators', 50, 300),
    #     'max_depth': trial.suggest_int('max_depth', 5, 30),
    #     'min_samples_split': trial.suggest_int('min_samples_split', 2, 20),
    #     'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 10),
    #     'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2']),
    #     'criterion': trial.suggest_categorical('criterion', ['gini', 'entropy', 'log_loss'])
    # }
    params = {
        'n_estimators': trial.suggest_int('n_estimators', 50, 200),  # Fewer trees
        'max_depth': trial.suggest_int('max_depth', 10, 25),         # Narrower
        'min_samples_split': trial.suggest_int('min_samples_split', 2, 10),
        'min_samples_leaf': trial.suggest_int('min_samples_leaf', 1, 5),
        'max_features': trial.suggest_categorical('max_features', ['sqrt', 'log2'])  # Drop None
    }
    model = RandomForestClassifier(**params, random_state=RANDOM_STATE)
    model.fit(X_train_pr_n, y_train["department_id"])
    y_pred = model.predict(X_valid_pr_n)
    score = f1_score(y_valid["department_id"], y_pred, average="weighted")
    return score # type: ignore

In [ ]:
study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=100)

[I 2026-03-04 10:57:31,589] A new study created in memory with name: no-name-3ef619ab-0a10-484c-aa5e-5db0cf94dc69


[I 2026-03-04 10:57:46,470] Trial 0 finished with value: 0.5132916108565354 and parameters: {'n_estimators': 197, 'max_depth': 13, 'min_samples_split': 3, 'min_samples_leaf': 5, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.5132916108565354.
[I 2026-03-04 10:57:46,168] Trial 1 finished with value: 0.4631742148695613 and parameters: {'n_estimators': 94, 'max_depth': 11, 'min_samples_split': 6, 'min_samples_leaf': 3, 'max_features': 'log2'}. Best is trial 0 with value: 0.5132916108565354.
[I 2026-03-04 10:57:55,278] Trial 2 finished with value: 0.4992187054265973 and parameters: {'n_estimators': 121, 'max_depth': 12, 'min_samples_split': 3, 'min_samples_leaf': 3, 'max_features': 'sqrt'}. Best is trial 0 with value: 0.5132916108565354.
[I 2026-03-04 10:57:59,890] Trial 3 finished with value: 0.5138817051911175 and parameters: {'n_estimators': 123, 'max_depth': 23, 'min_samples_split': 2, 'min_samples_leaf': 1, 'max_features': 'log2'}. Best is trial 3 with value: 0.513881705191117

In [109]:
study.best_params

{'n_estimators': 163,
 'max_depth': 19,
 'min_samples_split': 2,
 'min_samples_leaf': 1,
 'max_features': 'sqrt'}